# Module 3a — Generate evaluation questions from your library

**Time:** ~10 min · **Output:** `workshop_outputs/03_questions.csv`, `workshop_outputs/03_answers.csv`

1. Read and chunk the same documents you uploaded to MyGPT
2. Ask an LLM to write one question + reference answer per sampled chunk
3. Add unanswerable questions
4. **Review the questions yourself**, then run them through MyGPT

In [1]:
import sys
from pathlib import Path
import re
import random

# Make the repo root importable whether Jupyter was started from the root or this folder
ROOT = Path.cwd() if (Path.cwd() / "common").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from common.mygpt_client import MyGPTClient, outputs_dir, load_fallback_pubmedqa  # noqa: E402
OUT = outputs_dir()

In [2]:
DOCS_DIR = ROOT / "data" / "BioASQ" / "pdfs_70"
N_CHUNKS = 20                              # number of questions to generate (reduce on slow laptops)
CHUNK_SIZE = 1500                          # characters per chunk used for question generation
SEED = 42

client = MyGPTClient.from_env()
print("Generator LLM:", client.llm, "via", client.ollama_url)

Generator LLM: llama3:latest via http://host.docker.internal:11434


## 1. Read and chunk documents

In [3]:
def read_text(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix in {".txt", ".md"}:
        return path.read_text(encoding="utf-8", errors="ignore")
    if suffix == ".pdf":
        from pypdf import PdfReader
        return "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)
    if suffix == ".docx":
        import docx
        return "\n".join(p.text for p in docx.Document(path).paragraphs)
    return ""

chunks = []
for path in sorted(DOCS_DIR.glob("*")):
    text = re.sub(r"\s+", " ", read_text(path)).strip()
    for start in range(0, len(text), CHUNK_SIZE):
        piece = text[start:start + CHUNK_SIZE]
        if len(piece) > 400:                      # skip tiny tail chunks
            chunks.append({"source_doc": path.name, "chunk": piece})

print(len(chunks), "chunks from", len({c["source_doc"] for c in chunks}), "documents")
random.seed(SEED)
sample = random.sample(chunks, min(N_CHUNKS, len(chunks)))

755 chunks from 70 documents


## 2. Generate question + reference answer per chunk
The prompt asks for **specific, self-contained** questions: a reader shouldn't need to see the chunk to understand the question.

In [4]:
import json

QGEN_PROMPT = """You are building a benchmark for a question-answering system over a document library.
Read the passage and write ONE question that:
- can be answered from this passage alone,
- is specific and self-contained (do not say "the passage", "this document", "the author"),
- a researcher in this field would realistically ask.
Then write a concise reference answer (1-2 sentences) using only the passage.

Return JSON: {{"question": "...", "answer": "..."}}

PASSAGE:
{chunk}
"""

rows = []
for i, item in enumerate(sample, 1):
    try:
        raw = client.ollama(QGEN_PROMPT.format(chunk=item["chunk"]), temperature=0.2, json_mode=True)
        qa = json.loads(raw)
        rows.append({"question": qa["question"].strip(), "reference_answer": qa["answer"].strip(),
                     "answerable": 1, "source_doc": item["source_doc"], "source_chunk": item["chunk"]})
        print(f"[{i}] {qa['question']}")
    except Exception as exc:
        print(f"[{i}] skipped: {exc}")

[1] What types of gastrointestinal cancers are associated with metabolic dysfunction-associated steatotic liver disease (MASLD)?
[2] What is the mean age of patients with chronic obstructive pulmonary disease (COPD) in the study?
[3] What are the most common symptoms of postinfectious irritable bowel syndrome (PI-IBS)?
[4] What is the main difference in size between WJSC-MT and FIB-MT?
[5] What is the effect of PTEN loss on glioma cells' ability to evade the immune system?
[6] What percentage of glioblastoma patients showed EGFR gene amplification?
[7] What is the minimum viral load required for vector mosquitoes to be infected with dengue virus?
[8] What is the outcome of the precancerous cell's uncontrolled proliferation in the third stage?
[9] What drives the recurrence of glioblastoma multiforme (GBM)?
[10] What percentage of participants had a diagnosis of Generalized Anxiety Disorder?
[11] What percentage of children born outside the hospital of residence had CMVc screening perfo

## 3. Add unanswerable questions
Write questions that look plausible for your field but that your documents **do not** answer. You need these to test whether HI catches hallucinations.

In [9]:
import pandas as pd

UNANSWERABLE = [
    "What is the boiling point of tungsten?",
    "Who won the 2018 FIFA World Cup?",
    "What dose of imatinib is recommended for adults with chronic myeloid leukemia?",   # ← replace with near-miss questions for YOUR domain
]
NOT_IN_LIBRARY = "The library does not contain this information."
rows += [{"question": q, "reference_answer": NOT_IN_LIBRARY, "answerable": 0,
          "source_doc": "", "source_chunk": ""} for q in UNANSWERABLE]

questions = pd.DataFrame(rows)
questions.insert(0, "question_id", range(1, len(questions) + 1))
questions.to_csv(OUT / "03_questions.csv", index=False)
questions[["question_id", "question", "reference_answer", "answerable"]]

,question_id,question,reference_answer,answerable
0,1,What types of gastrointestinal cancers are ass...,MASLD is associated with an elevated risk of v...,1
1,2,What is the mean age of patients with chronic ...,The mean age of patients with COPD in the stud...,1
2,3,What are the most common symptoms of postinfec...,Abdominal pain and diarrhea are the most commo...,1
3,4,What is the main difference in size between WJ...,WJSC-MT resulted to be larger than FIB-MT in a...,1
4,5,What is the effect of PTEN loss on glioma cell...,Loss of the PTEN tumor suppressor gene leads t...,1
5,6,What percentage of glioblastoma patients showe...,38% showed EGFR gene amplification.,1
6,7,What is the minimum viral load required for ve...,Viremia > 104 copies/ml can still infect vecto...,1
7,8,What is the outcome of the precancerous cell's...,The precancerous cell develops uncontrolled pr...,1
8,9,What drives the recurrence of glioblastoma mul...,"A shift to a mesenchymal phenotype, mediated b...",1
9,10,What percentage of participants had a diagnosi...,Forty percent (n=8) of participants had a diag...,1


## 4. ✋ Human review (do not skip)
Open `workshop_outputs/03_questions.csv` in a spreadsheet or the Jupyter CSV viewer:
- **delete** questions that are vague, trivial or wrong
- **edit** reference answers that are incomplete
- check that each question really is answerable from its `source_chunk`

Save the file, then run the next cell to reload it.

In [10]:
questions = pd.read_csv(OUT / "03_questions.csv")
print(len(questions), "questions after review;", int((questions.answerable == 0).sum()), "unanswerable")

23 questions after review; 3 unanswerable


## 5. Run every question through MyGPT

In [12]:
answers = []
for i, r in questions.iterrows():
    print(f"[{i + 1}/{len(questions)}] {r.question}")
    try:
        res = client.ask(r.question)
    except Exception as exc:
        print("   failed:", exc)
        res = {"error": str(exc)}
    answers.append({**r.to_dict(), **{k: v for k, v in res.items() if k != "question"}})

answers = pd.DataFrame(answers)
answers.to_csv(OUT / "03_answers.csv", index=False)
answers[["question_id", "question", "answer", "QRS", "ARS", "HI"]].head(10)

[1/23] What types of gastrointestinal cancers are associated with metabolic dysfunction-associated steatotic liver disease (MASLD)?
[2/23] What is the mean age of patients with chronic obstructive pulmonary disease (COPD) in the study?
[3/23] What are the most common symptoms of postinfectious irritable bowel syndrome (PI-IBS)?
[4/23] What is the main difference in size between WJSC-MT and FIB-MT?
[5/23] What is the effect of PTEN loss on glioma cells' ability to evade the immune system?
[6/23] What percentage of glioblastoma patients showed EGFR gene amplification?
[7/23] What is the minimum viral load required for vector mosquitoes to be infected with dengue virus?
[8/23] What is the outcome of the precancerous cell's uncontrolled proliferation in the third stage?
[9/23] What drives the recurrence of glioblastoma multiforme (GBM)?
[10/23] What percentage of participants had a diagnosis of Generalized Anxiety Disorder?
[11/23] What percentage of children born outside the hospital of r

,question_id,question,answer,QRS,ARS,HI
0,1,What types of gastrointestinal cancers are ass...,"According to the provided context, metabolic d...",49.0,100.0,35.0
1,2,What is the mean age of patients with chronic ...,The mean age of patients with chronic obstruct...,57.0,90.0,26.0
2,3,What are the most common symptoms of postinfec...,"According to the provided context, the most co...",45.0,100.0,28.0
3,4,What is the main difference in size between WJ...,The main difference in size between WJSC-MT an...,24.0,83.0,23.0
4,5,What is the effect of PTEN loss on glioma cell...,"According to the provided context, PTEN loss i...",69.0,98.0,30.0
5,6,What percentage of glioblastoma patients showe...,"According to the context, EGFR gene amplificat...",65.0,94.0,28.0
6,7,What is the minimum viral load required for ve...,"Based on the provided context, the minimum vir...",54.0,97.0,40.0
7,8,What is the outcome of the precancerous cell's...,The outcome of the precancerous cell's uncontr...,54.0,100.0,38.0
8,9,What drives the recurrence of glioblastoma mul...,"According to the provided context, the recurre...",70.0,90.0,31.0
9,10,What percentage of participants had a diagnosi...,"According to the context, 40% (n=8) of partici...",60.0,87.0,31.0


➡️ Next: **03b_label_and_judge.ipynb**